# Transform Drivers Data

1. Read bronze `drivers` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfbirth` → `date_of_birth`)
1. Concatenate `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform the value to Title Case
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `drivers` table

Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.drivers"
silver_table=f"{catalog_name}.{silver_schema}.drivers"

### Read bronze drivers table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
drivers_df = spark.read.table(bronze_table).filter(f.col("batch_id") == v_batch_id)
display(drivers_df)

### Keep only the columns required for analytics (Drop url column)

In [0]:
drivers_selected_df=drivers_df.drop("url")

### Standardise column names using snake_case (driverld → driver_ld, datefbirth → date_of_birth)

In [0]:
drivers_named_df=(
    drivers_selected_df
    .withColumnsRenamed(
        {
        "driverId":"driver_id",
        "dateOfBirth":"date_of_birth"
        }
        )
)


### 4. Concatenate name.givenName and name. fanilyName to create a new column called driver_name and transform the value to Title Case

In [0]:
from pyspark.sql import functions as f

In [0]:
driver_concat_df=(
    drivers_named_df
    .withColumn(
        "name",
        f.concat_ws(" ", f.col("name.givenName"), f.col("name.familyName"))
    )
    .withColumnRenamed(
        "name",
        "driver_name"
    )
    )
display(driver_concat_df)

- Concatenate name.givenName and name. fanilyName to create a new column called driver_name and transform the value to Title Case
- Transform values of column nationality to Title Case

In [0]:
driver_renamed_df=(
    driver_concat_df
    .withColumns(
        {
            "driver_name": f.initcap(f.col("driver_name")),
            "nationality": f.initcap(f.col("nationality")),
            }
    )
)
display(driver_renamed_df)


### 5. Remove duplicate records

In [0]:
driver_final_df=driver_renamed_df.dropDuplicates(["driver_id"])

### 7. Write the transformed data to silver drivers table

In [0]:
# (
#     driver_final_df
#     .write\
#     .format("delta")\
#     .mode("overwrite")\
#     .saveAsTable(silver_table)
# )

write_to_silver(
    input_df=driver_final_df,
    target_table=silver_table,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
spark.sql(f"select * from {silver_table}").display()